In [ ]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from lightgbm import LGBMRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance
#import featuretools as ft
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from autofeat import AutoFeatRegressor
import matplotlib.pyplot as plt
from src.preprocessing import scale_features, add_outlier_flags, target_feature_split, one_hot_encoding, create_labels_for_classification
from src.feature_engineering import (create_vocal_instrumental_ratio, create_energy_rhythm_interaction, create_moodscore_bins
                                    ,create_vocal_energy_ratio,create_energy_acoustic_ratio,create_mood_rhythm_interaction,create_live_track_interaction
                                    ,create_classification_model, run_classification_model)


In [36]:
# Input Data
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")
df_sample_submission = pd.read_csv("../data/raw/sample_submission.csv")

df_train_bkp = df_train.copy()
df_test_bkp = df_test.copy()
df_sample_submission_bkp = df_sample_submission.copy()

In [37]:
flag_autofeat = False

In [38]:
if flag_autofeat:
    X = df_train.drop(columns=["BeatsPerMinute", "id"])
    y = df_train["BeatsPerMinute"]

    X_sample = X.sample(150000, random_state=42)
    y_sample = y.loc[X_sample.index]

    afreg = AutoFeatRegressor(
        verbose=1, 
        featsel_runs=1 
    )

    #X_auto = afreg.fit_transform(X, y)
    X_auto = afreg.fit_transform(X_sample, y_sample)

    print("Original Shape:", X_sample.shape)
    print("Expanded Shape:", X_auto.shape)
    print(X_auto.head())

In [39]:
#preprocessing
df_train, scaler = scale_features(df_train, ['AudioLoudness','TrackDurationMs'])
df_train, _ = add_outlier_flags(df_train, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

df_test, _ = scale_features(df_test,  ['AudioLoudness','TrackDurationMs'], scaler)
df_test, _ = add_outlier_flags(df_test, ['RhythmScore','AudioLoudness','VocalContent','AcousticQuality','InstrumentalScore','LivePerformanceLikelihood','TrackDurationMs'])

In [40]:
# feature engineering
df_train = create_vocal_instrumental_ratio(df_train)
df_train = create_energy_rhythm_interaction(df_train)
df_train = create_moodscore_bins(df_train)
df_train = create_vocal_energy_ratio(df_train)
df_train = create_energy_acoustic_ratio(df_train)
df_train = create_mood_rhythm_interaction(df_train)
df_train = create_live_track_interaction(df_train)


df_test = create_vocal_instrumental_ratio(df_test)
df_test = create_energy_rhythm_interaction(df_test)
df_test = create_moodscore_bins(df_test)
df_test = create_vocal_energy_ratio(df_test)
df_test = create_energy_acoustic_ratio(df_test)
df_test = create_mood_rhythm_interaction(df_test)
df_test = create_live_track_interaction(df_test)

In [41]:
#also preprocessing
df_train, encoder = one_hot_encoding(df_train, 'MoodScore_bins')
df_test, _ = one_hot_encoding(df_test, 'MoodScore_bins', encoder)

In [42]:
#Definition of X and y
X_train, y_train = target_feature_split(df=df_train, target="BeatsPerMinute", exclude_cols=['id'])
X_test, y_test = target_feature_split(df=df_test, exclude_cols=['id'])

In [43]:
RANDOM_STATE = 42
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
version = "v1"

In [44]:
train_models = {
    "dummy": DummyRegressor(strategy="mean"),
    "ridge": Ridge(random_state=RANDOM_STATE),
    "rf": RandomForestRegressor(max_depth=12,min_samples_leaf=5, n_jobs=-1,random_state=RANDOM_STATE),
    "lgbm": LGBMRegressor(n_jobs=-1, random_state=RANDOM_STATE)
}

oof = {name: np.zeros(len(X_train)) for name in train_models.keys()}

models = {}

for model in list(train_models.keys()):
    if os.path.exists(f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy"):
        models[f'{model}'] = joblib.load(f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl")
        oof[f'{model}'] = np.load(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy")
        del train_models[f'{model}']

In [45]:
for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Validation:  index={val_index}")
    X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    for model in train_models.keys():
        print(f"Fitting: {model}")
        train_models[f"{model}"].fit(X_tr, y_tr)
        oof[f"{model}"][val_index] = train_models[f"{model}"].predict(X_val)

for model in train_models.keys():
    print(f"Dumping and saving oof: {model}")
    joblib.dump(train_models[model], f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl")
    np.save(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy", oof[f'{model}'])
    models[f"{model}"] = train_models[f"{model}"]

Fold 0:
  Train: index=[     0      1      3 ... 524159 524160 524162]
  Validation:  index=[     2      6      7 ... 524146 524161 524163]
Fold 1:
  Train: index=[     1      2      3 ... 524160 524161 524163]
  Validation:  index=[     0     11     16 ... 524153 524159 524162]
Fold 2:
  Train: index=[     0      1      2 ... 524161 524162 524163]
  Validation:  index=[     4     10     12 ... 524148 524154 524155]
Fold 3:
  Train: index=[     0      2      4 ... 524161 524162 524163]
  Validation:  index=[     1      3      8 ... 524147 524157 524160]
Fold 4:
  Train: index=[     0      1      2 ... 524161 524162 524163]
  Validation:  index=[     5     13     15 ... 524151 524156 524158]


In [ ]:
#baseline rmse's and mae's
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.466341374596727 MAE: 21.19792095039579
#rf RMSE: 26.465788481708348 MAE: 21.197653869516085
#lgbm RMSE: 26.467350811600735 MAE: 21.198498087056862

#first try with new engineered features
#baseline rmse's and mae's
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.466341374596727 MAE: 21.19792095039579
#rf RMSE: 26.465788481708348 MAE: 21.197653869516085
#lgbm RMSE: 26.467350811600735 MAE: 21.198498087056862

#second try
#dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
#ridge RMSE: 26.464581033700515 MAE: 21.196263361285556
#rf RMSE: 26.46161412666076 MAE: 21.19429907984696
#lgbm RMSE: 26.467051728056493 MAE: 21.197983771897544



#Meta OOF RMSE: 26.461258311104743


In [47]:

rmses = {}
for name, preds in oof.items():
    print(name, "RMSE:", root_mean_squared_error(y_train, preds), "MAE:", mean_absolute_error(y_train, preds))
    rmses

dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
ridge RMSE: 26.465385825000094 MAE: 21.197030571261287
rf RMSE: 26.465305023507657 MAE: 21.196963534692447
lgbm RMSE: 26.467526741672962 MAE: 21.198457247155215


In [48]:
dict_importances = {}
for model in models.keys():
    print(model)
    _filepath = f"../data/processed/{model}_pi_{version}_seed{RANDOM_STATE}.parquet"
    if os.path.exists(_filepath):
        dict_importances[model] = pd.read_parquet(_filepath)
    else:
        perm = permutation_importance(models[model], X_train, y_train, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
        importance_df = pd.DataFrame({"feature": X_train.columns, "perm_importance": perm["importances_mean"]})
        importance_df = importance_df.sort_values("perm_importance", ascending=False)
        dict_importances[model] = importance_df
        importance_df.to_parquet(_filepath,index=False)

dummy
ridge
rf
lgbm


In [49]:
#stack = StackingRegressor(
#    estimators=[
#        ("ridge", models['ridge']),
#        ("rf", models['rf']),
#        ("lgbm", models['lgbm'])
#    ],
#    final_estimator=Ridge(),
#    passthrough=True,
#    n_jobs=-1
#)

#scores = cross_val_score(stack, X_train, y_train, cv=kf, scoring="neg_root_mean_squared_error")
#print("Stacking RMSE:", -np.mean(scores))


In [50]:
if not os.path.exists(f"../results/first_random_forest_submission.csv"):
    #Random Forest submission
    y_test_pred = models['rf'].predict(X_test)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_random_forest_submission.csv", index=False)
    #actually got worse results hehe

In [51]:
#Creating features with classification
y_train_clf = create_labels_for_classification(y_train)
X_train_v2, clf = create_classification_model(X_train, y_train_clf, kf, RANDOM_STATE)
y_train_v2 = y_train
version = "v2"

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.092821 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3841
[LightGBM] [Info] Number of data points in the train set: 524164, number of used features: 23
[LightGBM] [Info] Start training from score -1.609217
[LightGBM] [Info] Start training from score -0.510821
[LightGBM] [Info] Start training from score -1.609675


In [52]:
train_models_v2 = {
    "dummy": DummyRegressor(strategy="mean"),
    "ridge": Ridge(random_state=RANDOM_STATE),
    "rf": RandomForestRegressor(max_depth=12,min_samples_leaf=5, n_jobs=-1,random_state=RANDOM_STATE),
    "lgbm": LGBMRegressor(n_jobs=-1, random_state=RANDOM_STATE)
}

oof_v2 = {name: np.zeros(len(X_train_v2)) for name in train_models_v2.keys()}

models_v2 = {}

for model in list(train_models_v2.keys()):
    if os.path.exists(f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy"):
        models_v2[f'{model}'] = joblib.load(f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl")
        oof_v2[f'{model}'] = np.load(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy")
        del train_models_v2[f'{model}']


for i, (train_index, val_index) in enumerate(kf.split(X_train_v2, y_train_v2)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Validation:  index={val_index}")
    X_tr, X_val = X_train_v2.iloc[train_index], X_train_v2.iloc[val_index]
    y_tr, y_val = y_train_v2.iloc[train_index], y_train_v2.iloc[val_index]

    for model in train_models_v2.keys():
        print(f"Fitting: {model}")
        train_models_v2[f"{model}"].fit(X_tr, y_tr)
        oof_v2[f"{model}"][val_index] = train_models_v2[f"{model}"].predict(X_val)

for model in train_models_v2.keys():
    print(f"Dumping and saving oof: {model}")
    joblib.dump(train_models_v2[model], f"../models/{model}_{version}_seed{RANDOM_STATE}.pkl")
    np.save(f"../data/processed/{model}_oof_{version}_seed{RANDOM_STATE}.npy", oof_v2[f'{model}'])
    models_v2[f"{model}"] = train_models_v2[f"{model}"]

Fold 0:
  Train: index=[     0      1      3 ... 524159 524160 524162]
  Validation:  index=[     2      6      7 ... 524146 524161 524163]
Fold 1:
  Train: index=[     1      2      3 ... 524160 524161 524163]
  Validation:  index=[     0     11     16 ... 524153 524159 524162]
Fold 2:
  Train: index=[     0      1      2 ... 524161 524162 524163]
  Validation:  index=[     4     10     12 ... 524148 524154 524155]
Fold 3:
  Train: index=[     0      2      4 ... 524161 524162 524163]
  Validation:  index=[     1      3      8 ... 524147 524157 524160]
Fold 4:
  Train: index=[     0      1      2 ... 524161 524162 524163]
  Validation:  index=[     5     13     15 ... 524151 524156 524158]


In [53]:
dict_importances = {}
for model in models_v2.keys():
    print(model)
    _filepath = f"../data/processed/{model}_pi_{version}_seed{RANDOM_STATE}.parquet"
    if os.path.exists(_filepath):
        dict_importances[model] = pd.read_parquet(_filepath)
    else:
        perm = permutation_importance(models_v2[model], X_train_v2, y_train_v2, scoring="neg_root_mean_squared_error", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
        importance_df = pd.DataFrame({"feature": X_train_v2.columns, "perm_importance": perm["importances_mean"]})
        importance_df = importance_df.sort_values("perm_importance", ascending=False)
        dict_importances[model] = importance_df
        importance_df.to_parquet(_filepath,index=False)


dummy
ridge
rf
lgbm


In [54]:
dict_importances['ridge']

,feature,perm_importance
0,mood_rhythm_interaction,4.902001e-03
1,MoodScore_bins_low,4.075755e-03
2,live_track_interaction,2.163961e-03
3,MoodScore_bins_medium,1.474552e-03
4,proba_slow,1.048913e-03
5,vocal_energy_ratio,6.839092e-04
6,VocalContent,6.778439e-04
7,proba_fast,5.926350e-04
8,MoodScore,5.392569e-04
9,Energy,1.686728e-04


In [55]:
dict_importances['rf']

,feature,perm_importance
0,proba_slow,7.415966e-02
1,live_track_interaction,6.717005e-02
2,MoodScore,6.488419e-02
3,mood_rhythm_interaction,5.109140e-02
4,TrackDurationMs,4.725735e-02
5,RhythmScore,3.889076e-02
6,AudioLoudness,3.528426e-02
7,proba_fast,3.378924e-02
8,energy_acoustic_ratio,3.347493e-02
9,LivePerformanceLikelihood,3.212411e-02


In [56]:
dict_importances['lgbm']

,feature,perm_importance
0,MoodScore,0.047583
1,VocalContent,0.045234
2,mood_rhythm_interaction,0.042097
3,vocal_energy_ratio,0.034778
4,proba_slow,0.033145
5,live_track_interaction,0.032940
6,Energy,0.030247
7,RhythmScore,0.029947
8,energy_rhythm_interaction,0.029509
9,TrackDurationMs,0.027525


In [57]:
rmses = {}
for name, preds in oof_v2.items():
    print(name, "RMSE:", root_mean_squared_error(y_train_v2, preds), "MAE:", mean_absolute_error(y_train_v2, preds))
    rmses

dummy RMSE: 26.468078417587844 MAE: 21.19987341149104
ridge RMSE: 26.464581033700515 MAE: 21.196263361285556
rf RMSE: 26.46161412666076 MAE: 21.19429907984696
lgbm RMSE: 26.467051728056493 MAE: 21.197983771897544


In [58]:
if not os.path.exists(f"../results/second_random_forest_submission.csv"):
    #Random Forest submission
    X_test_v2 = run_classification_model(X_test, clf, random_state=RANDOM_STATE)
    y_test_pred = models_v2['rf'].predict(X_test_v2)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/second_random_forest_submission.csv", index=False)
    #actually got worse results hehe

In [59]:
dict_error_by_bin = {}
for model in models_v2.keys():
    df_resid = pd.DataFrame({"y_true": y_train_v2, "y_pred": oof_v2[model]})
    
    df_resid["BPM_bin"] = pd.qcut(df_resid["y_true"], q=5, labels=[f"Q1", "Q2", "Q3", "Q4", "Q5"])
    
    error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
    dict_error_by_bin[model] = error_by_bin

C:\Users\menez\AppData\Local\Temp\ipykernel_16492\816519922.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
C:\Users\menez\AppData\Local\Temp\ipykernel_16492\816519922.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_by_bin = df_resid.groupby("BPM_bin").apply(lambda d: root_mean_squared_error(d.y_true, d.y_pred))
C:\Users\menez\AppData\Local\Temp\ipykernel_16492\816519922.py:7: FutureWarning: The d

In [60]:
dict_error_by_bin

{'dummy': BPM_bin
 Q1    38.537388
 Q2    15.007049
 Q3     3.951164
 Q4    14.614545
 Q5    39.538506
 dtype: float64,
 'ridge': BPM_bin
 Q1    38.522524
 Q2    15.013749
 Q3     3.981901
 Q4    14.616621
 Q5    39.534894
 dtype: float64,
 'rf': BPM_bin
 Q1    38.498686
 Q2    15.024274
 Q3     4.060065
 Q4    14.629971
 Q5    39.531302
 dtype: float64,
 'lgbm': BPM_bin
 Q1    38.509849
 Q2    15.037905
 Q3     4.084525
 Q4    14.631043
 Q5    39.530529
 dtype: float64}

In [61]:
#be aware that this training takes a LOT OF TIME, so, if you don't already have the pkl, consider skipping this one. Just flag the skip_flag true.
#The baseline models can be created running "baseline.ipynb"
skip_flag = False
models_stacking = {}
if not os.path.exists(f"../models/stacking_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl") and skip_flag:
    
    models_stacking['ridge'] = joblib.load(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['rf'] = joblib.load(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['lgbm'] = joblib.load(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl")

    
    stack = StackingRegressor(
        estimators=[
            ("ridge", models_stacking['ridge']),
            ("rf", models_stacking['rf']),
            ("lgbm", models_stacking['lgbm'])
        ],
        final_estimator=Ridge(),
        passthrough=True,
        n_jobs=-1
    )

    stack.fit(X_train_v2, y_train_v2)
    models_stacking['stacking'] = stack

    y_test_pred = models_stacking['stacking'].predict(X_test)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stacking_submission.csv", index=False)
elif os.path.exists(f"../models/stacking_baseline_seed{RANDOM_STATE}.pkl"):
    models_stacking['stacking'] = joblib.load(f"../models/stacking_baseline_seed{RANDOM_STATE}.pkl")

    X_test_v2 = run_classification_model(X_test, clf, random_state=RANDOM_STATE)
    y_test_pred = models_stacking['stacking'].predict(X_test_v2)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stacking_submission.csv", index=False)

In [62]:
y_pred_train = models_stacking['stacking'].predict(X_train_v2)

df_resid = pd.DataFrame({"y_true": y_train_v2, "y_pred": y_pred_train})
df_resid["BPM_bin"] = pd.qcut(df_resid["y_true"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(
    lambda d: root_mean_squared_error(d.y_true, d.y_pred)
)
print(error_by_bin_in_sample)

BPM_bin
Q1    38.319606
Q2    14.934149
Q3     3.981366
Q4    14.553953
Q5    39.329343
dtype: float64


C:\Users\menez\AppData\Local\Temp\ipykernel_16492\4181536264.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(
C:\Users\menez\AppData\Local\Temp\ipykernel_16492\4181536264.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(


In [63]:
#be aware that this training takes a LOT OF TIME, so, if you don't already have the pkl, consider skipping this one. Just flag the skip_flag true.
#The baseline models can be created running "baseline.ipynb"
#Also, this .pkl is gigantic, and doesnt performe well, so, it is better to just ignore this one
skip_flag = False
models_stacking = {}
if not os.path.exists(f"../models/stackrf_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl") and skip_flag:
    
    models_stacking['ridge'] = joblib.load(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['rf'] = joblib.load(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['lgbm'] = joblib.load(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl")

    
    stack_rf = StackingRegressor(
        estimators=[
            ("ridge", models_stacking['ridge']),
            ("rf", models_stacking['rf']),
            ("lgbm", models_stacking['lgbm'])
        ],
        final_estimator=RandomForestRegressor(),
        passthrough=True,
        n_jobs=-1
    )

    stack_rf.fit(X_train_v2, y_train_v2)
    models_stacking['stackrt'] = stack_rf

    y_test_pred = models_stacking['stackrf'].predict(X_test)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stackrf_submission.csv", index=False)
elif os.path.exists(f"../models/stackrf_baseline_seed{RANDOM_STATE}.pkl") and skip_flag:
    models_stacking['stackrf'] = joblib.load(f"../models/stackrf_baseline_seed{RANDOM_STATE}.pkl")

    X_test_v2 = run_classification_model(X_test, clf, random_state=RANDOM_STATE)
    y_test_pred = models_stacking['stackrf'].predict(X_test_v2)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stackrf_submission.csv", index=False)

In [64]:
if skip_flag:
    y_pred_train = models_stacking['stackrf'].predict(X_train_v2)

    df_resid = pd.DataFrame({"y_true": y_train_v2, "y_pred": y_pred_train})
    df_resid["BPM_bin"] = pd.qcut(df_resid["y_true"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

    error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(
        lambda d: root_mean_squared_error(d.y_true, d.y_pred)
    )
    print(error_by_bin_in_sample)

In [65]:
skip_flag = False
models_stacking = {}
if not os.path.exists(f"../models/stacklgbm_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl") and os.path.exists(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl") and skip_flag:
    
    models_stacking['ridge'] = joblib.load(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['rf'] = joblib.load(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl")
    models_stacking['lgbm'] = joblib.load(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl")

    
    stack_lgbm = StackingRegressor(
        estimators=[
            ("ridge", models_stacking['ridge']),
            ("rf", models_stacking['rf']),
            ("lgbm", models_stacking['lgbm'])
        ],
        final_estimator=LGBMRegressor(),
        passthrough=True,
        n_jobs=-1
    )

    stack_lgbm.fit(X_train_v2, y_train_v2)
    models_stacking['stacklgbm'] = stack_lgbm

    y_test_pred = models_stacking['stacklgbm'].predict(X_test)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stacklgbm_submission.csv", index=False)
elif os.path.exists(f"../models/stacklgbm_baseline_seed{RANDOM_STATE}.pkl"):
    models_stacking['stacklgbm'] = joblib.load(f"../models/stacklgbm_baseline_seed{RANDOM_STATE}.pkl")

    X_test_v2 = run_classification_model(X_test, clf, random_state=RANDOM_STATE)
    y_test_pred = models_stacking['stacklgbm'].predict(X_test_v2)
    df_sample_submission['BeatsPerMinute'] = y_test_pred
    df_sample_submission.to_csv("../results/first_stacklgbm_submission.csv", index=False)

w:\Git\Estudos\Predicting-the-Beats-per-Minute-of-Songs-Kaggle\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [66]:
y_pred_train = models_stacking['stacklgbm'].predict(X_train_v2)

df_resid = pd.DataFrame({"y_true": y_train_v2, "y_pred": y_pred_train})
df_resid["BPM_bin"] = pd.qcut(df_resid["y_true"], q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])

error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(
    lambda d: root_mean_squared_error(d.y_true, d.y_pred)
)
print(error_by_bin_in_sample)

w:\Git\Estudos\Predicting-the-Beats-per-Minute-of-Songs-Kaggle\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


BPM_bin
Q1    38.394869
Q2    14.968791
Q3     4.046828
Q4    14.608424
Q5    39.431534
dtype: float64


C:\Users\menez\AppData\Local\Temp\ipykernel_16492\887609781.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(
C:\Users\menez\AppData\Local\Temp\ipykernel_16492\887609781.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  error_by_bin_in_sample = df_resid.groupby("BPM_bin").apply(


In [67]:
oof

{'dummy': array([119.03904198, 119.02395681, 119.05655411, ..., 119.05655411,
        119.03904198, 119.05655411]),
 'ridge': array([118.71584764, 119.09190103, 119.21303971, ..., 119.26900613,
        119.31121226, 119.54957502]),
 'rf': array([119.08213156, 118.7194301 , 119.12190676, ..., 118.67056506,
        120.02649325, 119.2110395 ]),
 'lgbm': array([118.85493373, 118.83321351, 119.53996441, ..., 119.02393038,
        120.5725371 , 119.03161934])}

In [68]:
oof_v2

{'dummy': array([119.03904198, 119.02395681, 119.05655411, ..., 119.05655411,
        119.03904198, 119.05655411]),
 'ridge': array([118.70064447, 118.83407309, 119.27151123, ..., 119.15839143,
        119.77917492, 119.4233583 ]),
 'rf': array([118.69863684, 118.71554808, 118.94435401, ..., 118.75434862,
        120.07189889, 119.39696764]),
 'lgbm': array([119.14729289, 118.76845879, 119.13932081, ..., 119.73164431,
        119.79813057, 119.37665611])}

In [72]:
X_train_baseline = pd.read_parquet("../data/processed/X_train_baseline.parquet")
y_train_baseline = pd.read_parquet("../data/processed/y_train_baseline.parquet")['BeatsPerMinute']
X_test_baseline = pd.read_parquet("../data/processed/X_test_baseline.parquet")

In [78]:
oof_df_baseline = pd.DataFrame({
    "ridge": np.load(f"../data/processed/ridge_oof_baseline_seed{RANDOM_STATE}.npy"),
    "rf": np.load(f"../data/processed/rf_oof_baseline_seed{RANDOM_STATE}.npy"),
    "lgbm": np.load(f"../data/processed/lgbm_oof_baseline_seed{RANDOM_STATE}.npy"),
})

meta = Ridge(alpha=1.0, random_state=RANDOM_STATE)
meta.fit(oof_df_baseline, y_train_baseline)
train_meta_pred = meta.predict(oof_df_baseline)
print("Meta OOF RMSE:", root_mean_squared_error(y_train_baseline, train_meta_pred))
joblib.dump(meta, f"../models/meta_ridge_with_oof_seed{RANDOM_STATE}.pkl")

models_baseline = {}
models_baseline["ridge"] = joblib.load(f"../models/ridge_baseline_seed{RANDOM_STATE}.pkl")
models_baseline["rf"] = joblib.load(f"../models/rf_baseline_seed{RANDOM_STATE}.pkl")
models_baseline["lgbm"] = joblib.load(f"../models/lgbm_baseline_seed{RANDOM_STATE}.pkl")

test_meta = pd.DataFrame({
    "ridge": models_baseline["ridge"].predict(X_test_baseline),
    "rf": models_baseline["rf"].predict(X_test_baseline),
    "lgbm": models_baseline["lgbm"].predict(X_test_baseline),
})

y_test_pred = meta.predict(test_meta)
df_sample_submission['BeatsPerMinute'] = y_test_pred
df_sample_submission.to_csv("../results/first_manual_stacking_submission.csv", index=False)

Meta OOF RMSE: 26.461258311104743


In [80]:
y_blend_test_simple = (models_baseline["ridge"].predict(X_test_baseline) + models_baseline["rf"].predict(X_test_baseline) + models_baseline["lgbm"].predict(X_test_baseline) + y_test_pred) / 4

In [81]:
df_sample_submission['BeatsPerMinute'] = y_blend_test_simple
df_sample_submission.to_csv("../results/first_blend_submission.csv", index=False)

In [69]:
##Stacking submission
#y_test_pred = stack.predict(X_test)
#df_sample_submission['BeatsPerMinute'] = y_test_pred
#df_sample_submission.to_csv("../results/first_stack_submission.csv", index=False)